In [6]:

import os
import numpy as np
import pandas as pd
import diptest

from sklearn.mixture import GaussianMixture

INPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_thickness_summary.csv"
OUTPUT_CSV = r"C:\Users\ishin\OneDrive\Desktop\ish\gbm_multimodality_summary.csv"

if not os.path.exists(INPUT_CSV):
    raise FileNotFoundError(
        f"Input file not found:\n{INPUT_CSV}"
    )

df = pd.read_csv(INPUT_CSV)

print("GBM MULTIMODALITY ANALYSIS")
print("=" * 60)

print(f"Loaded {len(df)} membrane components")
print(f"Patients : {df['patient_id'].nunique()}")

#FIT GMM MODELS
def fit_gmm_models(values):

    X = values.reshape(-1, 1)

    models = {}
    bic = {}
    aic = {}

    for n in [1, 2, 3]:

        model = GaussianMixture(
            n_components=n,
            random_state=42,
            n_init=20
        )

        model.fit(X)

        models[n] = model
        bic[n] = model.bic(X)
        aic[n] = model.aic(X)

    return models, bic, aic
results = []

for patient_id, group in df.groupby("patient_id"):

    print("\n" + "-" * 80)
    print(f"Patient : {patient_id}")

    thickness = (
        group["median_thickness_nm"]
        .dropna()
        .to_numpy()
    )

    n_samples = len(thickness)

    print(
        f"Membrane components : "
        f"{n_samples}"
    )

    if n_samples < 5:

        print(
            "Skipped (too few membrane components)"
        )

        continue


   # HARTIGAN'S DIP TEST
    dip_statistic, dip_pvalue = diptest.diptest(
        thickness
    )
    if dip_pvalue < 0.05:

        dip_classification = "Multimodal"

    else:

        dip_classification = "Unimodal"


    # GAUSSIAN MIXTURE MODELS
        models, bic, aic = fit_gmm_models(
        thickness
    )

    best_components = min(
        bic,
        key=bic.get
    )
    best_model = models[best_components]

    peak_positions = (
        best_model.means_.flatten()
    )

    component_weights = (
        best_model.weights_.flatten()
    )
   
    order = np.argsort(
        peak_positions
    )

    peak_positions = peak_positions[order]

    component_weights = component_weights[order]

    peaks = np.pad(
        peak_positions,
        (0, 3 - len(peak_positions)),
        constant_values=np.nan
    )

    weights = np.pad(
        component_weights,
        (0, 3 - len(component_weights)),
        constant_values=np.nan
    )

    if best_components >= 2:

        peak_distance = (
            peaks[1] - peaks[0]
        )

    else:

        peak_distance = np.nan


    if dip_classification == "Unimodal":

        if best_components == 1:

            gmm_interpretation = (
                "Unimodal; single GMM component"
            )

        else:

            gmm_interpretation = (
                "Unimodal; GMM subcomponents"
            )

    else:

        if best_components == 1:

            gmm_interpretation = (
                "Multimodal by Dip Test; "
                "single GMM component"
            )

        elif best_components == 2:

            gmm_interpretation = (
                "Multimodal"
            )

        else:

            gmm_interpretation = (
                "Multimodal"
            )


   # MULTIMODALITY STRENGTH
    if dip_pvalue < 0.01:

        if best_components >= 3:

            evidence = "Very Strong"

        elif best_components >= 2:

            evidence = "Strong"

        else:

            evidence = "Strong"

    elif dip_pvalue < 0.05:

        if best_components >= 2:

            evidence = "Strong"

        else:

            evidence = "Moderate"

    else:

        evidence = "None"

    print(
        f"Dip p-value          : "
        f"{dip_pvalue:.5f}"
    )

    print(
        f"Dip classification   : "
        f"{dip_classification}"
    )

    print(
        f"Best GMM             : "
        f"{best_components} component(s)"
    )

    print(
        f"BIC                  : "
        f"{bic[best_components]:.2f}"
    )

    print(
        f"AIC                  : "
        f"{aic[best_components]:.2f}"
    )

    print(
        f"Peak positions       : "
        f"{np.round(peaks, 2)}"
    )

    print(
        f"Peak weights         : "
        f"{np.round(weights, 3)}"
    )

    print(
        f"Peak separation      : "
        f"{peak_distance:.2f}"
        if not np.isnan(peak_distance)
        else "Peak separation      : N/A"
    )

    print(
        f"GMM interpretation   : "
        f"{gmm_interpretation}"
    )

    print(
        f"Evidence             : "
        f"{evidence}"
    )

    results.append({

        "Patient_ID":
            patient_id,

        "Number_of_membranes":
            n_samples,
       
        "Dip_statistic":
            dip_statistic,

        "Dip_p_value":
            dip_pvalue,

        "Dip_classification":
            dip_classification,

        
        "Best_GMM_components":
            best_components,

        "BIC":
            bic[best_components],

        "AIC":
            aic[best_components],

        
        "Peak_1_nm":
            peaks[0],

        "Peak_2_nm":
            peaks[1],

        "Peak_3_nm":
            peaks[2],

        "Weight_1":
            weights[0],

        "Weight_2":
            weights[1],

        "Weight_3":
            weights[2],

        
        "Peak_separation_nm":
            peak_distance,

        
        "GMM_interpretation":
            gmm_interpretation,

        "Evidence":
            evidence
    })

summary_df = pd.DataFrame(results)


if summary_df.empty:

    print(
        "\nNo valid patients were analysed."
    )

    raise SystemExit

summary_df = summary_df.sort_values(

    by=[
        "Dip_p_value",
        "Best_GMM_components",
        "BIC"
    ],

    ascending=[
        True,
        False,
        True
    ]

).reset_index(drop=True)

summary_df.insert(

    0,

    "Rank",

    np.arange(
        1,
        len(summary_df) + 1
    )
)

try:

    summary_df.to_csv(
        OUTPUT_CSV,
        index=False
    )

    print("\nResults saved successfully to:")
    print(OUTPUT_CSV)

except PermissionError:

    print(
        "\nWARNING: The output CSV is currently open "
        "or locked by another program."
    )

    print(
        "Please close the existing CSV file and run "
        "the following save command again:"
    )

    print()

    print(
        "summary_df.to_csv("
        "OUTPUT_CSV, index=False)"
    )


print("\n")

print(
    "PATIENT MULTIMODALITY RANKING"
)

print("=" * 60)


display_columns = [

    "Rank",

    "Patient_ID",

    "Dip_p_value",

    "Dip_classification",

    "Best_GMM_components",

    "GMM_interpretation",

    "Evidence",

    "Peak_1_nm",

    "Peak_2_nm",

    "Peak_3_nm"
]

print(

    summary_df[
        display_columns
    ].to_string(index=False)

)

print("\n")

print("SUMMARY")

print("=" * 60)


print(
    f"Patients analysed        : "
    f"{len(summary_df)}"
)

multimodal_count = (

    summary_df[
        "Dip_classification"
    ] == "Multimodal"

).sum()


unimodal_count = (

    summary_df[
        "Dip_classification"
    ] == "Unimodal"

).sum()


print(
    f"Multimodal patients      : "
    f"{multimodal_count}"
)


print(
    f"Unimodal patients        : "
    f"{unimodal_count}"
)

print("\n")

print("GMM CHARACTERISATION")

print("=" * 60)


gmm_1 = (
    summary_df[
        "Best_GMM_components"
    ] == 1
).sum()


gmm_2 = (
    summary_df[
        "Best_GMM_components"
    ] == 2
).sum()


gmm_3 = (
    summary_df[
        "Best_GMM_components"
    ] == 3
).sum()


print(
    f"Patients with 1 GMM component : "
    f"{gmm_1}"
)

print(
    f"Patients with 2 GMM components : "
    f"{gmm_2}"
)

print(
    f"Patients with 3 GMM components : "
    f"{gmm_3}"
)

print("\n")

print(
    "PATIENTS WITH GMM SUBCOMPONENTS "
    "BUT UNIMODAL DIP TEST"
)

print("=" * 60)


unimodal_with_multiple_gmm = summary_df[

    (summary_df["Dip_classification"] == "Unimodal")
    &
    (summary_df["Best_GMM_components"] > 1)

]


if len(unimodal_with_multiple_gmm) == 0:

    print(
        "None"
    )

else:

    print(

        unimodal_with_multiple_gmm[
            [
                "Patient_ID",
                "Dip_p_value",
                "Best_GMM_components",
                "GMM_interpretation"
            ]
        ].to_string(index=False)

    )

print("\n")

print(
    "Analysis completed successfully."
)

print("=" * 60)



GBM MULTIMODALITY ANALYSIS
Loaded 394 membrane components
Patients : 11

--------------------------------------------------------------------------------
Patient : 01-24
Membrane components : 19
Dip p-value          : 0.85855
Dip classification   : Unimodal
Best GMM             : 3 component(s)
BIC                  : 204.32
AIC                  : 196.76
Peak positions       : [170.83 379.97 480.62]
Peak weights         : [0.842 0.053 0.105]
Peak separation      : 209.14
GMM interpretation   : Unimodal; GMM subcomponents
Evidence             : None

--------------------------------------------------------------------------------
Patient : 02-24
Membrane components : 41
Dip p-value          : 0.99148
Dip classification   : Unimodal
Best GMM             : 3 component(s)
BIC                  : 477.29
AIC                  : 463.58
Peak positions       : [ 241.67  790.25 1336.31]
Peak weights         : [0.951 0.024 0.024]
Peak separation      : 548.58
GMM interpretation   : Unimodal; GMM sub